In [68]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_score

pd.set_option('display.max_columns', None)
prefix = "../data/processed/"

#Suppressing np error warnings, which have cropped up. The model always converges.
np.seterr(over='ignore', divide='ignore', invalid='ignore')

{'divide': 'ignore', 'over': 'ignore', 'under': 'ignore', 'invalid': 'ignore'}

In [69]:
#Regular games 
df = (pd.read_csv(prefix + "TeamStatisticsFrom2010RegRoll10.csv")).dropna()


In [70]:
#Merging home and away teams by gameId 
df_m = df.merge(
    df,
    on='gameId',
    suffixes=('_home', '_away')
)

# Keep only the home team's perspective (home == 1) to avoid mirrored duplicates
df_m = df_m[df_m['home_home'] == 1]
df_m = df_m[df_m['teamId_home'] != df_m['teamId_away']]

#Generating a list of columns from the differences in the rolling averages from the home and away teams
roll_fea = [x for x in df.columns if 'roll' in x]

for col in roll_fea: 
    diff_name = f'{col}_diff'
    df_m[diff_name] = df_m[f'{col}_home'] - df_m[f'{col}_away']


In [71]:
#Initial Test and Train split
xg_tr, xg_te, yg_tr, yg_te = train_test_split(df_m, df_m['win_home'], random_state=17, test_size=0.2)

In [72]:
#PyGAM initialization 
import scipy.sparse
from pygam import LogisticGAM, s, l, te

def to_array(self):
    return self.toarray()

scipy.sparse.spmatrix.A = property(to_array)

#Features for the pyGam
py_fea = ['fieldGoalsPercentage_roll10_diff', 'plusMinusPoints_roll10_diff', 'plusMinusPoints_roll10_away', 
          'win_roll10_diff', 'turnovers_roll10_home']


#Pipeline for pygam
pipe_g = Pipeline(steps=[
    ('scale', StandardScaler()),
    ('gam', LogisticGAM(
        s(0, 100, lam=10) + l(1) + l(2) + s(3, lam=10) + l(4) + te(0,1, lam=10), max_iter=300 
        ))
])


In [73]:
#KFold Validation
#Loop that iterates over each column in the above list, fits, and calculates the accuracy

kf       = KFold(n_splits=5)
stat_ary = np.zeros((3, 5))

for i, (train_ix, test_ix) in enumerate(kf.split(xg_tr)):
   xr = xg_tr[py_fea].iloc[train_ix].values
   yr = yg_tr.iloc[train_ix]
   xh = xg_tr[py_fea].iloc[test_ix].values
   yh = yg_tr.iloc[test_ix]
   pipe_g.fit(xr,yr)

   y_pred = pipe_g['gam'].predict(xh)
   acc    = 1.0*sum(y_pred==yh) / len(y_pred)
   prob   = pipe_g['gam'].predict_proba(xh)
   roc    = roc_auc_score(yh, prob)
   prec   = precision_score(yh, y_pred)

   stat_ary[0][i] = acc
   stat_ary[1][i] = roc
   stat_ary[2][i] = prec

print('\n\nKFold Results\nAccuracy  Mean(STD) %.3f(%.3f)\nROC       Mean(STD) %.3f(%.3f)\nPrecision Mean(STD) %.3f(%.3f)'%(stat_ary[0].mean(), stat_ary[0].std(), stat_ary[1].mean(), stat_ary[1].std(),stat_ary[2].mean(), stat_ary[2].std()))





KFold Results
Accuracy  Mean(STD) 0.632(0.008)
ROC       Mean(STD) 0.678(0.009)
Precision Mean(STD) 0.700(0.017)


In [74]:
#Full model test
pipe_g.fit(xg_tr[py_fea],yg_tr)

y_pred = pipe_g['gam'].predict(xg_te[py_fea])
acc    = 1.0*sum(y_pred==yg_te) / len(y_pred)
prob   = pipe_g['gam'].predict_proba(xg_te[py_fea])
roc    = roc_auc_score(yg_te, prob)
prec   = precision_score(yg_te, y_pred)

print('Full Model Results\nAccuracy  %.3f\nROC       %.3f\nPrecision %.3f'%(acc, roc, prec))

Full Model Results
Accuracy  0.624
ROC       0.675
Precision 0.687


In [75]:
#Cross model comparision data set
prefix = "../data/processed/"
matchups = pd.read_csv(prefix + 'TeamStatisticsFrom2010RegMatchups.csv')

In [76]:
#Season by season results
stat_ary = np.zeros((3,10))
j = 0
for i in range(2015, 2025):
    matchups_tt = df_m[df_m["season_home"].isin(range(i - 5, i))]
    matchups_val = df_m[df_m["season_home"] == i]
    X_tt = matchups_tt[py_fea]
    y_tt = matchups_tt['win_home']
    X_val = matchups_val[py_fea]
    y_val = matchups_val['win_home']

    pipe_g.fit(X_tt, y_tt)
    y_pred = pipe_g['gam'].predict(X_val)
    acc    = 1.0*sum(y_pred==y_val) / len(y_pred)
    prob   = pipe_g['gam'].predict_proba(X_val)
    roc    = roc_auc_score(y_val, prob)
    prec   = precision_score(y_val, y_pred)

    stat_ary[0][j] = acc
    stat_ary[1][j] = roc
    stat_ary[2][j] = prec
    j+=1

In [77]:
#Result dataframe construction and printing
yr = [x for x in range(2015,2025)]
df_comp = pd.DataFrame(stat_ary.T, index=yr, columns=['Accuracy', 'ROC', 'Precision'])

#Printing all results
print('Season by Season Results\n',df_comp)
print('\n\nMeans')
print(df_comp.mean())
print('\n\nStandard Deviations')
print(df_comp.std())



Season by Season Results
       Accuracy       ROC  Precision
2015  0.654472  0.706645   0.712660
2016  0.606504  0.655910   0.677273
2017  0.621951  0.675668   0.693271
2018  0.620325  0.675168   0.724315
2019  0.604396  0.658447   0.676533
2020  0.602778  0.652467   0.666667
2021  0.615638  0.655710   0.662752
2022  0.573171  0.618391   0.673394
2023  0.631707  0.690218   0.688928
2024  0.653061  0.704755   0.708117


Means
Accuracy     0.618400
ROC          0.669338
Precision    0.688391
dtype: float64


Standard Deviations
Accuracy     0.024338
ROC          0.026918
Precision    0.020847
dtype: float64
